# Comparing Morphological Characteristics of HCC1806 Fusion vs Control Clones

## Import Necessary Modules

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional, Dict
from pathlib import Path
import re
from collections import defaultdict

## Functions

In [ ]:
def stats_and_plot(dataframe, alpha=0.05, xlabel=None, ylabel=None, title=None, sig_bars=True, point_label=False, fig_size=(10, 6), font_sz=18, save_svg=False):
    """
    Perform t-tests between data in each column of the DataFrame
    and create a box and whisker plot with indicators for statistical significance.

    Parameters:
    dataframe (pd.DataFrame): The input DataFrame.
    alpha (float, optional): The significance level for the t-tests. Default is 0.05.
    save_svg (bool): If True, saves the plot as an SVG file using the title as the filename.

    Returns:
    pd.DataFrame: A DataFrame containing the p-values between each pair of columns.
    """
    import matplotlib as mpl
    from scipy.stats import shapiro, levene, mannwhitneyu, ttest_ind, kruskal
    from statsmodels.stats.multicomp import pairwise_tukeyhsd
    from scipy.stats import f_oneway, tukey_hsd
    import scikit_posthocs as sp
    import pingouin as pg

    dataframe = dataframe.apply(pd.to_numeric, errors='coerce')
    num_samples = dataframe.shape[1]
    p_values_df = pd.DataFrame(columns=dataframe.columns, index=dataframe.columns)

    fig, ax = plt.subplots(figsize=fig_size)

    # --- Clean spine styling ---
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_linewidth(1.5)
    ax.yaxis.set_ticks_position('left')
    ax.xaxis.set_ticks_position('bottom')

    # --- Boxplot ---
    bp = ax.boxplot(
        [dataframe[col].dropna() for col in dataframe.columns],
        labels=dataframe.columns,
        showmeans=True,
        showfliers=False,
        patch_artist=True,
        meanprops=dict(marker='D', markerfacecolor='black', markeredgecolor='black', markersize=5),
        medianprops=dict(color='black', linewidth=2),
        boxprops=dict(facecolor='white', color='black', linewidth=1.5),
        whiskerprops=dict(color='black', linewidth=1.5, linestyle='--'),
        capprops=dict(color='black', linewidth=1.5),
    )

    # --- Jittered data points ---
    for i, col in enumerate(dataframe.columns, start=1):
        y = dataframe[col].dropna()
        x = np.random.normal(i, 0.04, len(y))
        ax.scatter(x, y, color='dimgray', alpha=0.6, s=30, zorder=3, edgecolors='none')

        if point_label:
            valid_indices = dataframe[col].dropna().index
            for x_val, y_val, label in zip(x, y, valid_indices):
                ax.text(x_val, y_val, str(label), fontsize=8, ha='right', va='bottom')

    # --- Assumption testing ---
    alpha_assumption = 0.05
    groups = [dataframe[col].dropna().values for col in dataframe.columns]

    normality_ok = all(shapiro(g).pvalue > alpha_assumption for g in groups if len(g) >= 3)
    levene_p = levene(*groups).pvalue
    variance_ok = levene_p > alpha_assumption

    print(f"Normality (Shapiro-Wilk):     {'PASS' if normality_ok else 'FAIL'}")
    print(f"Equal variance (Levene's):    {'PASS' if variance_ok else 'FAIL'}")

    # --- Statistics ---
    if num_samples == 2:
        col1, col2 = dataframe.columns[0], dataframe.columns[1]
        g1, g2 = dataframe[col1].dropna().values, dataframe[col2].dropna().values

        if normality_ok and variance_ok:
            test_used = "Independent samples t-test"
            _, p_value = ttest_ind(g1, g2, equal_var=True)
        elif normality_ok and not variance_ok:
            test_used = "Welch's t-test"
            _, p_value = ttest_ind(g1, g2, equal_var=False)
        else:
            test_used = "Mann-Whitney U test"
            _, p_value = mannwhitneyu(g1, g2, alternative='two-sided')

        p_values_df.loc[col1, col2] = p_value
        print(f"Test selected:                {test_used}")
        print(f"p-value:                      {p_value:.4f}")

    else:
        if normality_ok and variance_ok:
            test_used = "One-way ANOVA + Tukey HSD"
            _, p_omnibus = f_oneway(*groups)
            res = tukey_hsd(*groups)
            p_values_df = pd.DataFrame(res.pvalue, columns=dataframe.columns, index=dataframe.columns)

        elif normality_ok and not variance_ok:
            test_used = "Welch's ANOVA + Games-Howell"
            melted = dataframe.melt(var_name='group', value_name='value').dropna()
            welch_result = pg.welch_anova(data=melted, dv='value', between='group')
            p_omnibus = welch_result['p_unc'].values[0]
            gh = pg.pairwise_gameshowell(data=melted, dv='value', between='group')
            p_values_df = pd.DataFrame(np.nan, columns=dataframe.columns, index=dataframe.columns)
            for _, row in gh.iterrows():
                p_values_df.loc[row['A'], row['B']] = row['pval']
                p_values_df.loc[row['B'], row['A']] = row['pval']

        else:
            test_used = "Kruskal-Wallis + Dunn's post-hoc (Bonferroni)"
            _, p_omnibus = kruskal(*groups)
            melted = dataframe.melt(var_name='group', value_name='value').dropna()
            dunn = sp.posthoc_dunn(melted, val_col='value', group_col='group', p_adjust='bonferroni')
            p_values_df = dunn

        print(f"Test selected:                {test_used}")
        print(f"Omnibus p-value:              {p_omnibus:.2e}")

    # --- Significance bars ---
    if sig_bars:
        y_max = dataframe.max().max()
        y_min_ax, y_max_ax = ax.get_ylim()
        y_span = y_max_ax - y_min_ax
        line_offset = y_span * 0.03
        text_offset = y_span * 0.01
        drawn_pairs = set()
        highest_y = y_max

        for col1 in p_values_df.columns:
            for col2 in p_values_df.index:
                p_value = p_values_df.loc[col2, col1]
                if not pd.isna(p_value) and p_value < alpha:
                    if (col1, col2) in drawn_pairs or (col2, col1) in drawn_pairs:
                        continue

                    if p_value < 0.001:
                        sig_symbol = '***'
                    elif p_value < 0.01:
                        sig_symbol = '**'
                    elif p_value < 0.05:
                        sig_symbol = '*'
                    else:
                        continue

                    x1 = dataframe.columns.get_loc(col1) + 1
                    x2 = dataframe.columns.get_loc(col2) + 1
                    y = y_max + line_offset

                    ax.plot([x1, x2], [y, y], lw=1.5, color='black')
                    ax.text((x1 + x2) * 0.5, y + text_offset, sig_symbol,
                            ha='center', va='bottom', color='black', fontsize=13)

                    drawn_pairs.add((col1, col2))
                    drawn_pairs.add((col2, col1))
                    line_offset += y_span * 0.09
                    highest_y = y + text_offset

        # Expand y-axis to fit all bars with a small buffer
        ax.set_ylim(bottom=dataframe.min().min() * 0.95, top=highest_y * 1.02)
        
    # --- Labels & title ---
    ax.set_xlabel(xlabel if xlabel else 'Samples', labelpad=10)
    ax.set_ylabel(ylabel if ylabel else 'Growth Rate', labelpad=10)
    plot_title = title if title else 'Boxplot with Statistical Significance Indicators'
    ax.set_title(plot_title, pad=12, fontweight='bold')

    fig.tight_layout()

    if save_svg:
        filename = f"{plot_title}.svg"
        fig.savefig(filename, format='svg', bbox_inches='tight', dpi=300)

    plt.show()
    return p_values_df

In [ ]:
# Matches e.g. "KH2404_F1_NEWavgnuclearsize_12hr_GFP.txt" or "KH2404_C1_avgnuclearsize_12hr_MC.txt"
# "sample" = whatever sits between "KH2404_" and "(NEW)avgnuclearsize", so it correctly
# captures multi-part names like "GFP_P" and "MC_P" for the parental clones.
NUCLEAR_FILENAME_PATTERN = re.compile(
    r"KH2404_(?P<sample>.+?)_(?:NEW)?avgnuclearsize_(?P<hr>\d+)hr_(?P<color>GFP|MC)\.txt$"
)

def read_nuclear_txt(filepath):
    """Read one Incucyte nuclear-area export and return its row-averaged value.

    Locates the header row by finding the line starting with 'Date Time' rather
    than assuming a fixed number of metadata lines, and excludes any 'Std Err'
    columns so they aren't averaged in with the actual value columns.
    """
    with open(filepath, "r", encoding="utf-8", errors="replace") as fh:
        lines = fh.readlines()

    header_idx = next(
        (i for i, line in enumerate(lines) if line.startswith("Date Time")),
        None,
    )
    if header_idx is None:
        raise ValueError(f"Could not find 'Date Time' header row in {filepath}")

    df = pd.read_csv(filepath, skiprows=header_idx, delimiter="\t", header=0)
    df = df.iloc[:, 2:]  # drop Date Time / Elapsed columns

    # Drop any "Std Err" columns, only average the actual value column(s)
    value_cols = [c for c in df.columns if "std err" not in c.lower()]
    df = df[value_cols]

    return df.iloc[0].mean()

def infer_group(sample: str, custom_group_map: Optional[Dict[str, str]] = None):
    if custom_group_map and sample in custom_group_map:
        return custom_group_map[sample]
    if sample.startswith("F"):
        return "Fusion"
    if sample.startswith("C"):
        return "Control"
    if sample.startswith("GFP") or sample.startswith("MC"):
        return "Parental"
    return None


def load_nuclear_area_directory(
    directory: str,
    groups=("Fusion", "Control", "Parental"),
    custom_group_map: Optional[Dict[str, str]] = None,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Scan `directory` for all Incucyte nuclear-area .txt exports, average
    together any files that belong to the same sample (e.g. the GFP + MC
    files for a fusion clone), and return a DataFrame shaped like
    cell_area_df: index = sample, columns = group (Fusion/Control/Parental).
    """
    directory = Path(directory)
    files = sorted(directory.glob("*.txt"))
    if not files:
        raise FileNotFoundError(f"No .txt files found in {directory}")

    sample_values = defaultdict(list)
    sample_files = defaultdict(list)

    for f in files:
        m = NUCLEAR_FILENAME_PATTERN.match(f.name)
        if not m:
            print(f"  Skipping unrecognized filename: {f.name}")
            continue
        sample = m.group("sample")
        value = read_nuclear_txt(f)
        sample_values[sample].append(value)
        sample_files[sample].append(f.name)

    nuclear_df = pd.DataFrame(index=sorted(sample_values.keys()), columns=list(groups), dtype=float)

    for sample, values in sample_values.items():
        group = infer_group(sample, custom_group_map)
        if group is None:
            print(f"  Could not infer group for sample '{sample}', skipping")
            continue
        nuclear_df.loc[sample, group] = np.mean(values)
        if verbose:
            n = len(values)
            tag = f"averaged {n} files" if n > 1 else "1 file"
            print(f"{sample:10s} -> {group:9s} ({tag}: {', '.join(sample_files[sample])})")

    return nuclear_df

## Main Function

### Cell Area Measurements Data Import

In [ ]:
file = '/stor/work/Brock/kennedy/SC_repo/data/Morphology/HCC1806_Morphology/cell_area.csv'

raw_data_file_path = file

cell_area_df = pd.read_csv(raw_data_file_path, delimiter=',', index_col=0)  # Change delimiter if necessary

In [ ]:
area_p_values = stats_and_plot(cell_area_df[['Control','Fusion']], alpha=0.05, font_sz=24, sig_bars=True, point_label=False, fig_size=(6, 6), ylabel='Cell Area (μm²)',title='Comparing Cell Area for HCC1806 Control vs Fusion Clones',save_svg=True)

print(area_p_values)
print(cell_area_df)

print("Cell Area for Samples by Group")
for column in cell_area_df.columns:
    average = cell_area_df[column].mean()
    standard_error = cell_area_df[column].sem()
    standard_dev = cell_area_df[column].std()
    print(column)
    print("Average:", average)
    print("Standard Error:", standard_error)
    print("Standard Deviation:", standard_dev,'\n')

### Cell Nuclear Measurements

In [ ]:
nuclear_df = load_nuclear_area_directory('/stor/work/Brock/kennedy/SC_repo/data/Morphology/HCC1806_Morphology/nuclear_area_bothcolors/')

In [ ]:
nuclear_p_values = stats_and_plot(nuclear_df[['Control','Fusion']], alpha=0.05, sig_bars=True, font_sz=24, point_label=False, fig_size=(6, 6), ylabel='Nuclear Area (μm²)',title='Comparing Nuclear Area for HCC1806 Control vs Fusion Clones',save_svg=True)

print(nuclear_p_values)
print(nuclear_df)

print("Nuclear Area for Samples by Group")
for column in nuclear_df.columns:
    average = nuclear_df[column].mean()
    standard_error = nuclear_df[column].sem()
    standard_dev = nuclear_df[column].std()
    print(column)
    print("Average:", average)
    print("Standard Error:", standard_error)
    print("Standard Deviation:", standard_dev,'\n')

### Nuclear to Cytoplasmic Ratio

In [ ]:
nuc_cyto_ratio = nuclear_df/cell_area_df

nuc_cyto_p_values = stats_and_plot(nuc_cyto_ratio[['Control','Fusion']], alpha=0.05, font_sz=24, sig_bars=True, point_label=False, fig_size=(6, 6), ylabel='Ratio (Nuclear Area/Cell Area)',title='HCC1806 NC ratio\n ',save_svg=True)

print(nuc_cyto_p_values)
print(nuc_cyto_ratio)

print("Nuclear to Cytoplasmic Ratio for Samples by Group")
for column in nuc_cyto_ratio.columns:
    average = nuc_cyto_ratio[column].mean()
    standard_error = nuc_cyto_ratio[column].sem()
    standard_dev = nuc_cyto_ratio[column].std()
    print(column)
    print("Average:", average)
    print("Standard Error:", standard_error)
    print("Standard Deviation:", standard_dev,'\n')